## 1 ) Loading Dataset

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

CSV_PATH = "./raw_loan_dataset.csv"
df = pd.read_csv(CSV_PATH)

OUT_PATH = "./clean_loan_dataset.csv"
print("\n first 5 Raws")
print(df.head())

print("\n information of data")
print(df.info())

print("\n Missing Values")
print(df.isnull().sum())



 first 5 Raws
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  

 information of data
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null

## 2) Clean currency formatting



In [13]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

## 3) Normalize categorical typos Before imputation

In [14]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "yEs": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  


## 4) Impute missing values

In [15]:
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

print("After imputing Missing Values" , df.isnull().sum())

After imputing Missing Values Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


## 5) Remove duplicates


In [16]:
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


## 6) IQR capping on numeric columns

In [17]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

## 7) Label encoding (Yes/No → 0/1)

In [18]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))


=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


## 8) Class balance check

In [19]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


## 9) Feature engineering (no leakage from label)

In [20]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)
print(df.tail())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
95  66359.0        541.0            12.55     38200.0              1   
96  61241.0        781.0            24.40     22200.0              0   
97  63829.0        596.0             9.30     16000.0              1   
98  97529.0        629.0            17.60     15900.0              0   
99  62268.0        720.0            12.20     20700.0              1   

    PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
95                 0         0      0.575657            4897.343173  
96                 1         0      0.362502            2411.062992  
97                 0         1      0.250670            6196.990291  
98                 0         0      0.163028            5243.494624  
99                 0         1      0.332434            4717.272727  


## 10) RobustScaler

#### I have used The Predicting file and I told the reason I have used there 

In [15]:
# binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
# numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
# scale_cols = [c for c in numeric_cols if c not in binary_cols]

# scaler = RobustScaler()
# df[scale_cols] = scaler.fit_transform(df[scale_cols])

## 11) Final snapshot + save

In [21]:
print("\n=== FINAL HEAD ===")
print(df.head())

print("\n=== FINAL INFO ===")
print(df.info())

print("\n=== FINAL MISSING VALUES ===")
print(df.isnull().sum())

CSV_PATH = "./raw_loan_dataset.csv"
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")


=== FINAL HEAD ===
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              1   
1   96482.0        524.0             15.0     11200.0              1   
2   28478.0        638.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              1   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.237111           51814.285714  
1                 0         1      0.116084            6030.125000  
2                 0         1      0.424889            4449.687500  
3                 0         1      0.270783            1389.838710  
4                 0         1      0.278675            3555.185185  

=== FINAL INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column          